# 3D CBCT Prognosis: Image + Tooth Metadata

Binary outcome:
- 0: Healed
- 1: Not-healed (Healing + Non-healed)

Inputs:
- 3D tooth-centered CBCT ROI
- tooth number from `Dataset A - Overview.xlsx`

The US/Universal tooth number is converted to:
- arch: maxillary vs mandibular
- tooth type: anterior vs premolar vs molar

The raw tooth number itself is not treated as a continuous predictor.

Automatic training mode:
- If `PRETRAINED_PATH` exists:
  - pretrained segmentation encoder LR = `1e-5`
  - new prognosis classifier LR = `1e-4`
- Otherwise:
  - whole network scratch LR = `1e-4`


In [17]:
# If needed:
# !pip install wandb scikit-learn scipy tqdm

from pathlib import Path
from collections import Counter
import random
import re

import nibabel as nib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import wandb

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from prognosis_dataset_tooth_metadata import PrognosisDataset
from prognosis_model_tooth_metadata import PrognosisModel


SEED = 42
DATA_DIR = Path("../DSApre/roi_crop")
OUTCOME_XLSX = Path("Dataset A - Overview.xlsx")

# If this checkpoint exists, the notebook automatically switches to
# pretrained fine-tuning mode. Otherwise it trains from scratch.
PRETRAINED_PATH = Path(
    "/storage/home/hcoda1/4/rchen438/r-jli3175-0/data/Dental_CBCT_3d/saved_big/exp_4po_2024-07-12-11-46-16/checkpoint_900.pt"
)
USE_PRETRAINED = (
    PRETRAINED_PATH is not None
    and PRETRAINED_PATH.exists()
)

TARGET_SHAPE = np.array([176, 160, 288])

NUM_CLASSES = 2
CLASS_NAMES = ["Healed", "Not-healed"]

USE_TOOTH_METADATA = True

# Model receives:
# [mandibular, anterior, premolar, molar]
TOOTH_METADATA_DIM = 4

BATCH_SIZE = 2
NUM_WORKERS = 2
NUM_EPOCHS = 300

# Learning rates
SCRATCH_LR = 1e-4
PRETRAINED_ENCODER_LR = 1e-5
CLASSIFIER_LR = 1e-4

WEIGHT_DECAY = 1e-4
DROPOUT = 0.3
PRELU = True

# Keep regularization mild for the small dataset.
LABEL_SMOOTHING = 0.0
GRAD_CLIP_NORM = 5.0

LR_SCHEDULER_FACTOR = 0.5
LR_SCHEDULER_PATIENCE = 4

MIN_SCRATCH_LR = 1e-6
MIN_ENCODER_LR = 1e-7
MIN_CLASSIFIER_LR = 1e-6

EARLY_STOPPING_PATIENCE = 30

# Conservative 3D augmentation.
# Random crop/resize is intentionally avoided because lesion/tooth size
# can carry prognostic information.
AUG_ROTATION_DEGREES = 180.0
AUG_TRANSLATION_VOXELS = 40.0
AUG_SPATIAL_PROB = 0.30

AUG_INTENSITY_PROB = 0.30
AUG_INTENSITY_SCALE_RANGE = (0.90, 1.10)
AUG_INTENSITY_SHIFT_FRACTION = 0.1

AUG_NOISE_PROB = 0.10
AUG_NOISE_STD_FRACTION = 0.05

CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
USE_AMP = DEVICE.type == "cuda"

print("Device:", DEVICE)
print("AMP:", USE_AMP)
print(
    "Training mode:",
    "PRETRAINED" if USE_PRETRAINED else "SCRATCH",
)

if USE_PRETRAINED:
    print("Pretrained checkpoint:", PRETRAINED_PATH)
    print("Encoder LR:", PRETRAINED_ENCODER_LR)
    print("Classifier LR:", CLASSIFIER_LR)
else:
    print("Scratch LR:", SCRATCH_LR)


def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)


seed_everything(SEED)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)


Device: cuda
AMP: True
Training mode: PRETRAINED
Pretrained checkpoint: /storage/home/hcoda1/4/rchen438/r-jli3175-0/data/Dental_CBCT_3d/saved_big/exp_4po_2024-07-12-11-46-16/checkpoint_900.pt
Encoder LR: 1e-05
Classifier LR: 0.0001


## 1. Discover ROI images and exclude oversized cases


In [18]:
dataset_all = PrognosisDataset(
    data_dir=DATA_DIR
)

excluded = [
    sample
    for sample in dataset_all.samples
    if np.any(
        np.asarray(sample["shape"])
        > TARGET_SHAPE
    )
]

print("Excluded oversized cases:", len(excluded))

for sample in excluded:
    print(
        sample["case_id"],
        tuple(sample["shape"]),
    )

dataset_all.samples = [
    sample
    for sample in dataset_all.samples
    if np.all(
        np.asarray(sample["shape"])
        <= TARGET_SHAPE
    )
]

dataset_all.target_shape = TARGET_SHAPE.copy()

print("\nRemaining image cases:", len(dataset_all.samples))
print("Target shape:", tuple(dataset_all.target_shape))


Excluded oversized cases: 7
DSA024pre (101, 89, 308)
DSA029pre (159, 136, 292)
DSA054pre (105, 115, 349)
DSA057pre (190, 137, 261)
DSA074pre (112, 122, 330)
DSA147pre (91, 78, 300)
DSA201pre (129, 165, 272)

Remaining image cases: 190
Target shape: (176, 160, 288)


## 2. Create prognosis labels and match them to available ROI images


In [19]:
df = pd.read_excel(
    OUTCOME_XLSX
)


def normalize_case_id(
    x,
):
    match = re.search(
        r"DSA[-_ ]?0*(\d+)",
        str(x),
        re.IGNORECASE,
    )

    if match is None:
        return None

    return (
        f"DSA"
        f"{int(match.group(1)):03d}"
    )


def extract_pai(
    x,
):
    match = re.search(
        r"\d+",
        str(x),
    )

    if match is None:
        return None

    return int(
        match.group()
    )


def assign_label(
    row,
):
    post_raw = str(
        row[
            "Follow-up CBCT-PAI [POST]"
        ]
    ).strip()

    if post_raw.lower() in {
        "-",
        "",
        "nan",
        "none",
    }:
        return None

    if (
        "extract"
        in post_raw.lower()
    ):
        return 2

    pre = extract_pai(
        row[
            "Pre-op CBCT-PAI [PRE]"
        ]
    )

    post = extract_pai(
        row[
            "Follow-up CBCT-PAI [POST]"
        ]
    )

    if post is None:
        return None

    # Healed
    if post <= 2:
        return 0

    if pre is None:
        return None

    # Healing
    if post < pre:
        return 1

    # Non-healed
    return 2


def normalize_column_name(
    x,
):
    return re.sub(
        r"[^a-z0-9]",
        "",
        str(x).lower(),
    )


def find_tooth_number_column(
    dataframe,
):
    """
    Find the tooth-number column without hard-coding one spelling.

    The selected column is printed so the mapping is auditable.
    """
    normalized = {
        normalize_column_name(col): col
        for col in dataframe.columns
    }

    exact_candidates = [
        "toothnumber",
        "toothno",
        "toothnum",
        "tooth",
        "toothid",
        "tooth#",
        "teethnumber",
    ]

    for candidate in exact_candidates:
        key = normalize_column_name(
            candidate
        )

        if key in normalized:
            return normalized[key]

    # Fallback for names such as
    # "Target Tooth Number" or "Interested Tooth #".
    for col in dataframe.columns:
        key = normalize_column_name(
            col
        )

        if (
            "tooth" in key
            and (
                "number" in key
                or "num" in key
                or "no" in key
            )
        ):
            return col

    raise KeyError(
        "Could not identify the tooth-number column in "
        "Dataset A - Overview.xlsx. Available columns:\\n"
        + "\\n".join(
            str(x)
            for x in dataframe.columns
        )
    )


TOOTH_NUMBER_COLUMN = find_tooth_number_column(
    df
)

print(
    "Using tooth-number column:",
    TOOTH_NUMBER_COLUMN,
)


df[
    "normalized_case_id"
] = (
    df["Sequence"]
    .apply(
        normalize_case_id
    )
)

df = df[
    df[
        "normalized_case_id"
    ].notna()
].copy()


df[
    "label"
] = df.apply(
    assign_label,
    axis=1,
)


# Convert tooth number to numeric.
df[
    "tooth_number"
] = pd.to_numeric(
    df[
        TOOTH_NUMBER_COLUMN
    ],
    errors="coerce",
)


df = df[
    df[
        "label"
    ].notna()
].copy()


df[
    "label"
] = (
    df["label"]
    .astype(int)
)


# Tooth number must follow US/Universal numbering.
valid_tooth = (
    df["tooth_number"]
    .between(
        1,
        32,
        inclusive="both",
    )
)

if USE_TOOTH_METADATA:
    n_invalid_tooth = int(
        (~valid_tooth).sum()
    )

    if n_invalid_tooth > 0:
        print(
            "Cases excluded because tooth number "
            "is missing/invalid:",
            n_invalid_tooth,
        )

    df = df[
        valid_tooth
    ].copy()


df[
    "tooth_number"
] = (
    df["tooth_number"]
    .astype(int)
)


label_map = dict(
    zip(
        df[
            "normalized_case_id"
        ],
        df[
            "label"
        ],
    )
)


tooth_number_map = dict(
    zip(
        df[
            "normalized_case_id"
        ],
        df[
            "tooth_number"
        ],
    )
)


prognosis_data = []


for sample in (
    dataset_all.samples
):
    normalized_id = (
        normalize_case_id(
            sample[
                "case_id"
            ]
        )
    )

    if (
        normalized_id
        not in label_map
    ):
        continue

    if (
        USE_TOOTH_METADATA
        and normalized_id
        not in tooth_number_map
    ):
        continue

    item = {
        "case_id":
            sample[
                "case_id"
            ],

        "image":
            str(
                sample[
                    "image"
                ]
            ),

        "label":
            int(
                label_map[
                    normalized_id
                ]
            ),
    }

    if USE_TOOTH_METADATA:
        item[
            "tooth_number"
        ] = int(
            tooth_number_map[
                normalized_id
            ]
        )

    prognosis_data.append(
        item
    )


print(
    "Usable cases:",
    len(
        prognosis_data
    ),
)


print(
    "Original classes:",
    Counter(
        x["label"]
        for x
        in prognosis_data
    ),
)


if USE_TOOTH_METADATA:
    tooth_table = pd.DataFrame(
        [
            {
                "case_id":
                    x["case_id"],

                "tooth_number":
                    x["tooth_number"],

                "arch":
                    PrognosisDataset
                    .tooth_metadata_from_number(
                        x["tooth_number"]
                    )[1],

                "tooth_type":
                    PrognosisDataset
                    .tooth_metadata_from_number(
                        x["tooth_number"]
                    )[2],
            }
            for x in prognosis_data
        ]
    )

    print(
        "\\nArch distribution:"
    )
    print(
        tooth_table[
            "arch"
        ].value_counts()
    )

    print(
        "\\nTooth type distribution:"
    )
    print(
        tooth_table[
            "tooth_type"
        ].value_counts()
    )


Using tooth-number column: Tooth Number [US]
Usable cases: 159
Original classes: Counter({0: 125, 2: 23, 1: 11})
\nArch distribution:
arch
Maxillary     83
Mandibular    76
Name: count, dtype: int64
\nTooth type distribution:
tooth_type
Molar       102
Premolar     51
Anterior      6
Name: count, dtype: int64


In [20]:
# PrognosisDataset discovery already prefers *_roi_img_refined.nii.gz
# when available, and dataset_all has already been filtered for oversized ROIs.
# Do not rebuild prognosis_data here because doing so would bypass that filtering.

print(
    "Using filtered prognosis_data:",
    len(prognosis_data),
    Counter(x["label"] for x in prognosis_data),
)


Using filtered prognosis_data: 159 Counter({0: 125, 2: 23, 1: 11})


## 3. Convert to the binary outcome and create the fixed train / validation / test split

The label conversion is performed **before** stratification:
- Healed → 0
- Healing or Non-healed → 1

The test set is held out and is not evaluated during training.


In [21]:
# Convert to binary before the split so stratification matches the actual task.
for item in prognosis_data:
    item["label"] = (
        0 if int(item["label"]) == 0 else 1
    )

print(
    "Binary class counts:",
    Counter(x["label"] for x in prognosis_data),
)

train_data, temp_data = train_test_split(
    prognosis_data,
    test_size=0.30,
    random_state=SEED,
    stratify=[
        x["label"]
        for x in prognosis_data
    ],
)

val_data, test_data = train_test_split(
    temp_data,
    test_size=0.50,
    random_state=SEED,
    stratify=[
        x["label"]
        for x in temp_data
    ],
)


def print_split(name, data):
    print(
        f"{name}: {len(data)}",
        Counter(
            x["label"]
            for x in data
        ),
    )


print_split("Train", train_data)
print_split("Val", val_data)
print_split("Test", test_data)


# Save case IDs for reproducibility.
split_rows = []

for split_name, split_data in [
    ("train", train_data),
    ("val", val_data),
    ("test", test_data),
]:
    for x in split_data:
        split_rows.append(
            {
                "split": split_name,
                "case_id": x["case_id"],
                "label": x["label"],
                "image": x["image"],
                "tooth_number": x.get("tooth_number"),
            }
        )

pd.DataFrame(
    split_rows
).to_csv(
    CHECKPOINT_DIR / "prelim_split_seed42.csv",
    index=False,
)


Binary class counts: Counter({0: 125, 1: 34})
Train: 111 Counter({0: 87, 1: 24})
Val: 24 Counter({0: 19, 1: 5})
Test: 24 Counter({0: 19, 1: 5})


In [22]:
# Sanity check after the split.
assert set(x["label"] for x in train_data).issubset({0, 1})
assert set(x["label"] for x in val_data).issubset({0, 1})
assert set(x["label"] for x in test_data).issubset({0, 1})

print("Binary labels verified.")


Binary labels verified.


## 4. Compute intensity statistics from the training split only

For the segmentation mask:
- use `*_roi_seg_refined.nii.gz` if available
- otherwise use `*_roi_seg.nii.gz`

The statistics use foreground voxels where `seg != 0`, matching the segmentation pretraining convention.


In [23]:
def get_train_stats(
    train_samples,
    data_dir,
    min_perc=0.05,
    max_perc=99.5,
):
    data_dir = Path(data_dir)

    fg_pixels = []

    for sample in tqdm(
        train_samples,
        desc="Computing train intensity stats",
    ):
        img_path = Path(
            sample["image"]
        )

        case_id = sample["case_id"]

        seg_original = (
            data_dir
            / f"{case_id}_roi_seg.nii.gz"
        )

        seg_refined = (
            data_dir
            / f"{case_id}_roi_seg_refined.nii.gz"
        )

        seg_path = (
            seg_refined
            if seg_refined.exists()
            else seg_original
        )

        if not seg_path.exists():
            print(f"Skipping {case_id}: segmentation not found")
            continue

        img = nib.load(
            img_path
        ).get_fdata().astype(
            np.float32
        )

        seg = nib.load(
            seg_path
        ).get_fdata()

        if img.shape != seg.shape:
            raise ValueError(
                f"Image/seg shape mismatch for {case_id}: "
                f"{img.shape} vs {seg.shape}"
            )

        fg = img[
            seg != 0
        ]

        if fg.size == 0:
            raise ValueError(
                f"Empty foreground segmentation: {case_id}"
            )

        fg_pixels.append(
            fg.astype(
                np.float32,
                copy=False,
            )
        )

    fg_pixels = np.concatenate(
        fg_pixels,
        axis=0,
    )

    stats = {
        "mean": float(
            np.mean(fg_pixels)
        ),
        "std": float(
            np.std(fg_pixels)
        ),
        "min": float(
            np.percentile(
                fg_pixels,
                min_perc,
            )
        ),
        "max": float(
            np.percentile(
                fg_pixels,
                max_perc,
            )
        ),
    }

    del fg_pixels

    return stats


train_stats = get_train_stats(
    train_data,
    DATA_DIR,
    min_perc=0.05,
    max_perc=99.5,
)

print(train_stats)


Computing train intensity stats:   0%|          | 0/111 [00:00<?, ?it/s]

{'mean': 2186.043212890625, 'std': 631.0716552734375, 'min': 805.0, 'max': 4095.0}


## 5. Build PyTorch datasets and dataloaders

Augmentation is enabled **only for the training set**. Validation and test images are deterministic.


In [24]:
train_ds = PrognosisDataset(
    data_dir=DATA_DIR,
    samples=train_data,
    train_stats=train_stats,
    target_shape=TARGET_SHAPE,
    augment=True,
    use_tooth_metadata=USE_TOOTH_METADATA,
    rotation_degrees=AUG_ROTATION_DEGREES,
    translation_voxels=AUG_TRANSLATION_VOXELS,
    spatial_aug_prob=AUG_SPATIAL_PROB,
    intensity_aug_prob=AUG_INTENSITY_PROB,
    intensity_scale_range=AUG_INTENSITY_SCALE_RANGE,
    intensity_shift_fraction=AUG_INTENSITY_SHIFT_FRACTION,
    noise_prob=AUG_NOISE_PROB,
    noise_std_fraction=AUG_NOISE_STD_FRACTION,
)

val_ds = PrognosisDataset(
    data_dir=DATA_DIR,
    samples=val_data,
    train_stats=train_stats,
    target_shape=TARGET_SHAPE,
    augment=False,
    use_tooth_metadata=USE_TOOTH_METADATA,
)

test_ds = PrognosisDataset(
    data_dir=DATA_DIR,
    samples=test_data,
    train_stats=train_stats,
    target_shape=TARGET_SHAPE,
    augment=False,
    use_tooth_metadata=USE_TOOTH_METADATA,
)


loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": (
        DEVICE.type == "cuda"
    ),
    "worker_init_fn": seed_worker,
    "persistent_workers": (
        NUM_WORKERS > 0
    ),
}

train_loader = DataLoader(
    train_ds,
    shuffle=True,
    generator=train_generator,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_ds,
    shuffle=False,
    **loader_kwargs,
)

test_loader = DataLoader(
    test_ds,
    shuffle=False,
    **loader_kwargs,
)

batch = next(
    iter(train_loader)
)

print(
    "Image batch:",
    batch["image"].shape,
)

print(
    "Label batch:",
    batch["label"],
)


if USE_TOOTH_METADATA:
    print(
        "Example tooth number:",
        batch["tooth_number"],
    )
    print(
        "Example tooth features "
        "[mandibular, anterior, premolar, molar]:",
        batch["tooth_features"],
    )


Image batch: torch.Size([2, 1, 176, 160, 288])
Label batch: tensor([1, 0])
Example tooth number: tensor([19,  4])
Example tooth features [mandibular, anterior, premolar, molar]: tensor([[1., 0., 0., 1.],
        [0., 0., 1., 0.]])


## 6. Build the prognosis model and load the pretrained segmentation encoder


In [25]:
model = PrognosisModel(
    in_channels=1,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT,
    metadata_dim=(
        TOOTH_METADATA_DIM
        if USE_TOOTH_METADATA
        else 0
    ),
    prelu=PRELU
)

# These are the layers transferred from the segmentation model.
if USE_PRETRAINED:
    checkpoint = torch.load(
        PRETRAINED_PATH,
        map_location="cpu",
    )

    state_dict = checkpoint.get(
        "model_state_dict",
        checkpoint,
    )

    # Remove DataParallel prefix if present.
    state_dict = {
        (
            key.replace("module.", "", 1)
            if key.startswith("module.")
            else key
        ): value
        for key, value in state_dict.items()
    }

    # Map the UNet checkpoint keys to the Prognosis model keys
    key_mapping = {
        "encoders.0.": "conv1.",
        "encoders.1.": "conv2.",
        "encoders.2.": "conv3.",
        "encoders.3.": "conv4.",
        "encoders.4.": "conv5.",
        "encoders.5.": "bottleneck.",  # Standard if bottleneck is part of the encoders list
        "bottleneck.": "bottleneck.",  # Fallback if bottleneck is named explicitly
    }

    encoder_state_dict = {}
    for key, value in state_dict.items():
        for old_prefix, new_prefix in key_mapping.items():
            if key.startswith(old_prefix):
                # Translate 'encoders.0.conv1.weight' -> 'conv1.conv1.weight'
                new_key = key.replace(old_prefix, new_prefix, 1)
                encoder_state_dict[new_key] = value
                break  # Stop checking prefixes once a match is found

    if len(encoder_state_dict) == 0:
        raise RuntimeError(
            "USE_PRETRAINED=True, but no encoder tensors matched "
            "the prognosis model. Check checkpoint key names."
        )

    load_result = model.load_state_dict(
        encoder_state_dict,
        strict=False,
    )

    print(
        "Loaded pretrained encoder tensors:",
        len(encoder_state_dict),
    )
    print(
        "Missing keys:",
        load_result.missing_keys,
    )
    print(
        "Unexpected keys:",
        load_result.unexpected_keys,
    )

else:
    print("Training the entire model from scratch.")

model = model.to(DEVICE)


Loaded pretrained encoder tensors: 54
Missing keys: ['classifier.1.weight', 'classifier.1.bias']
Unexpected keys: []


## 7. Loss, optimizer, and learning-rate schedule

The binary training set is imbalanced, so inverse-frequency class weights are used.

Additional standard regularization/training components:
- mild label smoothing
- AdamW with weight decay
- validation-loss-based learning-rate reduction
- gradient clipping in the training loop
- mixed precision when CUDA is available


The weighted loss is applied at the sample level because the 3D batch size is 1.  
A weighted sampler is not used simultaneously with weighted cross-entropy to avoid double compensation for class imbalance.


In [26]:
train_labels = np.array(
    [x["label"] for x in train_data],
    dtype=int,
)

class_counts = np.bincount(
    train_labels,
    minlength=NUM_CLASSES,
)

if np.any(class_counts == 0):
    raise ValueError(
        f"At least one training class is empty: {class_counts}"
    )

class_weights_np = (
    len(train_labels)
    / (
        NUM_CLASSES
        * class_counts
    )
)

class_weights = torch.tensor(
    class_weights_np,
    dtype=torch.float32,
    device=DEVICE,
)

print("Class counts:", class_counts)
print("Class weights:", class_weights_np)


def weighted_cross_entropy(
    logits,
    labels,
):
    losses = F.cross_entropy(
        logits,
        labels,
        reduction="none",
        label_smoothing=LABEL_SMOOTHING,
    )

    sample_weights = class_weights[
        labels
    ]

    return (
        losses
        * sample_weights
    ).mean()


# ------------------------------------------------------------
# Optimizer
# ------------------------------------------------------------
# Pretrained mode:
#   encoder      -> small LR
#   new classifier/head -> larger LR
#
# Scratch mode:
#   entire network -> one LR
# ------------------------------------------------------------

if USE_PRETRAINED:
    encoder_params = []
    classifier_params = []

    encoder_prefixes = (
        "conv1.",
        "conv2.",
        "conv3.",
        "conv4.",
        "conv5.",
        "bottleneck.",
    )

    encoder_params = []
    classifier_params = []

    for name, param in model.named_parameters():
        if name.startswith(encoder_prefixes):
            encoder_params.append(param)
        else:
            classifier_params.append(param)

    if len(encoder_params) == 0:
        raise RuntimeError(
            "No encoder parameters were found for differential LR."
        )

    if len(classifier_params) == 0:
        raise RuntimeError(
            "No classifier parameters were found for differential LR."
        )

    optimizer = torch.optim.AdamW(
        [
            {
                "params": encoder_params,
                "lr": PRETRAINED_ENCODER_LR,
                "name": "encoder",
            },
            {
                "params": classifier_params,
                "lr": CLASSIFIER_LR,
                "name": "classifier",
            },
        ],
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_SCHEDULER_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
        min_lr=[
            MIN_ENCODER_LR,
            MIN_CLASSIFIER_LR,
        ],
    )

else:
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=SCRATCH_LR,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=LR_SCHEDULER_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
        min_lr=MIN_SCRATCH_LR,
    )


print("Optimizer parameter groups:")
for i, group in enumerate(optimizer.param_groups):
    print(
        f"  group {i}: "
        f"{group.get('name', 'all')} | "
        f"lr={group['lr']:.2e} | "
        f"n_params={sum(p.numel() for p in group['params']):,}"
    )


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


Class counts: [87 24]
Class weights: [0.63793103 2.3125    ]
Optimizer parameter groups:
  group 0: encoder | lr=1.00e-05 | n_params=14,155,350
  group 1: classifier | lr=1.00e-04 | n_params=1,034


## 8. Metrics and train/evaluation functions

Primary monitoring for this imbalanced binary model:
- validation loss
- ROC AUC
- balanced accuracy
- macro F1

Accuracy is logged but is not used as the main model-selection criterion.


In [27]:
def compute_metrics(
    y_true,
    y_prob,
):
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    y_prob = np.asarray(
        y_prob,
        dtype=float,
    )

    y_pred = np.argmax(
        y_prob,
        axis=1,
    )

    metrics = {
        "accuracy": accuracy_score(
            y_true,
            y_pred,
        ),
        "balanced_acc": balanced_accuracy_score(
            y_true,
            y_pred,
        ),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
    }

    try:
        if NUM_CLASSES == 2:
            metrics["macro_auc"] = roc_auc_score(
                y_true,
                y_prob[:, 1],
            )
        else:
            metrics["macro_auc"] = roc_auc_score(
                y_true,
                y_prob,
                multi_class="ovr",
                average="macro",
                labels=list(range(NUM_CLASSES)),
            )
    except ValueError as e:
        print("AUC error:", e)
        print("y_true distribution:", Counter(y_true))
        print("y_prob shape:", y_prob.shape)
        print("Any NaN in probability:", np.isnan(y_prob).any())
        metrics["macro_auc"] = np.nan

    recalls = recall_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average=None,
        zero_division=0,
    )

    for class_idx, recall in enumerate(
        recalls
    ):
        metrics[
            f"recall_class_{class_idx}"
        ] = float(recall)

    return metrics


def train_one_epoch(
    model,
    loader,
    optimizer,
    scaler,
    epoch,
):
    model.train()

    running_loss = 0.0
    n_seen = 0
    y_true = []
    y_prob = []

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch:03d} [Train]",
        leave=False,
    )

    for batch in pbar:
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True,
        )

        tooth_features = (
            batch["tooth_features"].to(
                DEVICE,
                non_blocking=True,
            )
            if USE_TOOTH_METADATA
            else None
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.amp.autocast(
            "cuda",
            enabled=USE_AMP
        ):
            logits = model(
                images,
                tooth_features,
            )

            loss = weighted_cross_entropy(
                logits,
                labels,
            )

        scaler.scale(
            loss
        ).backward()

        # Unscale before clipping so the threshold is applied to true gradients.
        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=GRAD_CLIP_NORM,
        )

        scaler.step(
            optimizer
        )
        scaler.update()

        batch_size = images.size(0)
        n_seen += batch_size

        running_loss += (
            loss.item()
            * batch_size
        )

        probs = torch.softmax(
            logits.detach().float(),
            dim=1,
        )

        y_true.extend(
            labels.detach()
            .cpu()
            .numpy()
            .tolist()
        )

        y_prob.extend(
            probs.cpu()
            .numpy()
            .tolist()
        )

        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            avg=f"{running_loss / n_seen:.4f}",
        )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    metrics = compute_metrics(
        y_true,
        y_prob,
    )

    return (
        epoch_loss,
        metrics,
    )


@torch.no_grad()
def evaluate(
    model,
    loader,
    split_name="Val",
):
    model.eval()

    running_loss = 0.0
    y_true = []
    y_prob = []
    case_ids = []

    pbar = tqdm(
        loader,
        desc=f"[{split_name}]",
        leave=False,
    )

    for batch in pbar:
        images = batch["image"].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch["label"].to(
            DEVICE,
            non_blocking=True,
        )

        tooth_features = (
            batch["tooth_features"].to(
                DEVICE,
                non_blocking=True,
            )
            if USE_TOOTH_METADATA
            else None
        )

        with torch.amp.autocast(
            "cuda",
            enabled=USE_AMP
        ):
            logits = model(
                images,
                tooth_features,
            )

            loss = weighted_cross_entropy(
                logits,
                labels,
            )

        running_loss += (
            loss.item()
            * images.size(0)
        )

        probs = torch.softmax(
            logits.float(),
            dim=1,
        )

        y_true.extend(
            labels.cpu()
            .numpy()
            .tolist()
        )

        y_prob.extend(
            probs.cpu()
            .numpy()
            .tolist()
        )

        case_ids.extend(
            list(
                batch["case_id"]
            )
        )

        pbar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = (
        running_loss
        / len(loader.dataset)
    )

    metrics = compute_metrics(
        y_true,
        y_prob,
    )

    y_prob = np.asarray(
        y_prob
    )

    return {
        "loss": epoch_loss,
        "metrics": metrics,
        "y_true": np.asarray(y_true),
        "y_prob": y_prob,
        "y_pred": np.argmax(
            y_prob,
            axis=1,
        ),
        "case_ids": case_ids,
    }


## 9. Initialize Weights & Biases


In [28]:
run_name = (
    "prelim-pretrained-unet-2class-aug-tooth"
    if USE_PRETRAINED
    else "prelim-scratch-unet-2class-aug-tooth"
)

run = wandb.init(
    project="dental-prognosis",
    name=run_name,
    config={
        "seed": SEED,
        "training_mode": (
            "pretrained"
            if USE_PRETRAINED
            else "scratch"
        ),
        "use_pretrained": USE_PRETRAINED,
        "pretrained_path": (
            str(PRETRAINED_PATH)
            if USE_PRETRAINED
            else None
        ),
        "num_classes": NUM_CLASSES,
        "use_tooth_metadata": USE_TOOTH_METADATA,
        "tooth_metadata_dim": (
            TOOTH_METADATA_DIM
            if USE_TOOTH_METADATA
            else 0
        ),
        "tooth_metadata_definition": (
            "[mandibular, anterior, premolar, molar]"
            if USE_TOOTH_METADATA
            else None
        ),
        "class_names": CLASS_NAMES,
        "target_shape": TARGET_SHAPE.tolist(),
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,

        "scratch_lr": SCRATCH_LR,
        "pretrained_encoder_lr": PRETRAINED_ENCODER_LR,
        "classifier_lr": CLASSIFIER_LR,

        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
        "label_smoothing": LABEL_SMOOTHING,
        "grad_clip_norm": GRAD_CLIP_NORM,

        "lr_scheduler_factor": LR_SCHEDULER_FACTOR,
        "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,

        "augmentation": {
            "rotation_degrees": AUG_ROTATION_DEGREES,
            "translation_voxels": AUG_TRANSLATION_VOXELS,
            "spatial_prob": AUG_SPATIAL_PROB,
            "intensity_prob": AUG_INTENSITY_PROB,
            "intensity_scale_range": AUG_INTENSITY_SCALE_RANGE,
            "intensity_shift_fraction": AUG_INTENSITY_SHIFT_FRACTION,
            "noise_prob": AUG_NOISE_PROB,
            "noise_std_fraction": AUG_NOISE_STD_FRACTION,
        },

        "train_n": len(train_ds),
        "val_n": len(val_ds),
        "test_n": len(test_ds),

        "class_counts_train": class_counts.tolist(),
        "class_weights": class_weights_np.tolist(),
        "intensity_stats": train_stats,
    },
)


wandb: WARNING Tried to log to step 297 that is less than the current step 300. Steps must be monotonically increasing, so this data will be ignored. See https://wandb.me/define-metric to log data out of order.


## 10. Train

The best checkpoint is selected using validation loss.  
The test set is not touched here.


In [29]:
best_val_loss = float("inf")
best_epoch = -1
epochs_without_improvement = 0

mode_name = (
    "pretrained"
    if USE_PRETRAINED
    else "scratch"
)

best_path = (
    CHECKPOINT_DIR
    / f"best_{mode_name}_unet_2class_prognosis_aug_tooth.pth"
)


for epoch in range(
    1,
    NUM_EPOCHS + 1,
):
    train_loss, train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        epoch=epoch,
    )

    val_result = evaluate(
        model=model,
        loader=val_loader,
        split_name="Val",
    )

    val_loss = val_result[
        "loss"
    ]

    val_metrics = val_result[
        "metrics"
    ]

    scheduler.step(
        val_loss
    )

    if USE_PRETRAINED:
        encoder_lr = optimizer.param_groups[0]["lr"]
        classifier_lr = optimizer.param_groups[1]["lr"]
    else:
        encoder_lr = optimizer.param_groups[0]["lr"]
        classifier_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f} | "
        f"Val AUC: {val_metrics['macro_auc']:.4f} | "
        f"Val BalAcc: {val_metrics['balanced_acc']:.4f} | "
        f"Val Macro-F1: {val_metrics['macro_f1']:.4f} | "
        f"Encoder LR: {encoder_lr:.2e} | "
        f"Classifier LR: {classifier_lr:.2e}"
    )

    wandb.log(
        {
            "epoch": epoch,

            "train/loss": train_loss,
            "train/accuracy": train_metrics["accuracy"],
            "train/balanced_acc": train_metrics["balanced_acc"],
            "train/macro_f1": train_metrics["macro_f1"],
            "train/macro_auc": train_metrics["macro_auc"],

            "val/loss": val_loss,
            "val/accuracy": val_metrics["accuracy"],
            "val/balanced_acc": val_metrics["balanced_acc"],
            "val/macro_f1": val_metrics["macro_f1"],
            "val/macro_auc": val_metrics["macro_auc"],
            "val/recall_healed": val_metrics["recall_class_0"],
            "val/recall_not_healed": val_metrics["recall_class_1"],

            "lr/encoder": encoder_lr,
            "lr/classifier": classifier_lr,
        },
        step=epoch,
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch,
                "training_mode": mode_name,
                "use_pretrained": USE_PRETRAINED,
                "pretrained_path": (
                    str(PRETRAINED_PATH)
                    if USE_PRETRAINED
                    else None
                ),

                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),

                "best_val_loss": best_val_loss,
                "train_stats": train_stats,
                "target_shape": TARGET_SHAPE,
                "class_names": CLASS_NAMES,
                "class_weights": class_weights_np,
                "seed": SEED,

                "learning_rates": {
                    "scratch": SCRATCH_LR,
                    "pretrained_encoder": PRETRAINED_ENCODER_LR,
                    "classifier": CLASSIFIER_LR,
                },

                "augmentation": {
                    "rotation_degrees": AUG_ROTATION_DEGREES,
                    "translation_voxels": AUG_TRANSLATION_VOXELS,
                    "spatial_prob": AUG_SPATIAL_PROB,
                    "intensity_prob": AUG_INTENSITY_PROB,
                    "intensity_scale_range": AUG_INTENSITY_SCALE_RANGE,
                    "intensity_shift_fraction": AUG_INTENSITY_SHIFT_FRACTION,
                    "noise_prob": AUG_NOISE_PROB,
                    "noise_std_fraction": AUG_NOISE_STD_FRACTION,
                },
            },
            best_path,
        )

        print(
            f"  -> Saved best model: {best_path}"
        )

    else:
        epochs_without_improvement += 1

    if (
        epochs_without_improvement
        >= EARLY_STOPPING_PATIENCE
    ):
        print(
            f"Early stopping at epoch {epoch}. "
            f"Best epoch: {best_epoch}"
        )
        break

wandb.summary["best_epoch"] = best_epoch
wandb.summary["best_val_loss"] = best_val_loss


Epoch 001 [Train]:   0%|          | 0/56 [00:00<?, ?it/s]

[Val]:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 001/300 | Train loss: 0.6948 | Val loss: 0.6867 | Val AUC: 0.3895 | Val BalAcc: 0.3684 | Val Macro-F1: 0.3684 | Encoder LR: 1.00e-05 | Classifier LR: 1.00e-04
  -> Saved best model: checkpoints/best_pretrained_unet_2class_prognosis_aug_tooth.pth


Epoch 002 [Train]:   0%|          | 0/56 [00:00<?, ?it/s]

[Val]:   0%|          | 0/12 [00:00<?, ?it/s]

...

[Val]:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 299/300 | Train loss: 0.6043 | Val loss: 0.6469 | Val AUC: 0.6947 | Val BalAcc: 0.7158 | Val Macro-F1: 0.6250 | Encoder LR: 1.00e-07 | Classifier LR: 1.00e-06


Epoch 300 [Train]:   0%|          | 0/56 [00:00<?, ?it/s]

[Val]:   0%|          | 0/12 [00:00<?, ?it/s]

Epoch 300/300 | Train loss: 0.5986 | Val loss: 0.6470 | Val AUC: 0.6947 | Val BalAcc: 0.7158 | Val Macro-F1: 0.6250 | Encoder LR: 1.00e-07 | Classifier LR: 1.00e-06


## 11. Final held-out test evaluation

Run this only after training/model selection is complete.


In [30]:
best_checkpoint = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)

model.eval()

test_result = evaluate(
    model=model,
    loader=test_loader,
    split_name="Test",
)

test_metrics = test_result[
    "metrics"
]

print(
    f"Best epoch: {best_checkpoint['epoch']}"
)

print(
    f"Test loss: {test_result['loss']:.4f}"
)

print(
    f"Test macro AUC: {test_metrics['macro_auc']:.4f}"
)

print(
    f"Test balanced accuracy: {test_metrics['balanced_acc']:.4f}"
)

print(
    f"Test macro F1: {test_metrics['macro_f1']:.4f}"
)

print(
    f"Test accuracy: {test_metrics['accuracy']:.4f}"
)

print("\nPer-class recall:")

for idx, name in enumerate(
    CLASS_NAMES
):
    print(
        f"{name}: "
        f"{test_metrics[f'recall_class_{idx}']:.4f}"
    )

cm = confusion_matrix(
    test_result["y_true"],
    test_result["y_pred"],
    labels=list(
        range(NUM_CLASSES)
    ),
)

print("\nConfusion matrix:")
print(cm)


wandb.log(
    {
        "test/loss": test_result["loss"],
        "test/accuracy": test_metrics["accuracy"],
        "test/balanced_acc": test_metrics["balanced_acc"],
        "test/macro_f1": test_metrics["macro_f1"],
        "test/macro_auc": test_metrics["macro_auc"],
        #"test/recall_healed": test_metrics["recall_class_0"],
        #"test/recall_healing": test_metrics["recall_class_1"],
        #"test/recall_non_healed": test_metrics["recall_class_2"],
        "test/recall_healed": test_metrics["recall_class_0"],
        "test/recall_not_healed": test_metrics["recall_class_1"],
    },
    step=best_checkpoint["epoch"],
)

wandb.log(
    {
        "test/confusion_matrix": wandb.plot.confusion_matrix(
            probs=None,
            y_true=test_result["y_true"],
            preds=test_result["y_pred"],
            class_names=CLASS_NAMES,
        )
    }
)

prediction_df = pd.DataFrame(
    {
        "case_id": test_result["case_ids"],
        "tooth_number": [
            x.get("tooth_number")
            for x in test_data
        ],
        "label": test_result["y_true"],
        "prediction": test_result["y_pred"],
        "prob_healed": test_result["y_prob"][:, 0],
        "prob_not_healed": test_result["y_prob"][:, 1],
      #  "prob_non_healed": test_result["y_prob"][:, 2],
    }
)

prediction_path = (
    CHECKPOINT_DIR
    / "test_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False,
)

print(
    "\nSaved test predictions:",
    prediction_path,
)

wandb.finish()


[Test]:   0%|          | 0/12 [00:00<?, ?it/s]

Best epoch: 297
Test loss: 0.6802
Test macro AUC: 0.5789
Test balanced accuracy: 0.5579
Test macro F1: 0.4126
Test accuracy: 0.4167

Per-class recall:
Healed: 0.3158
Not-healed: 0.8000

Confusion matrix:
[[ 6 13]
 [ 1  4]]

Saved test predictions: checkpoints/test_predictions.csv


epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▆▆▇▇▇▇█████
lr/classifier,███████▄▄▃▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
lr/encoder,████████████▄▄▄▄▃▃▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/accuracy,█▇▇▄▆▄▄▃▄▂▁▄▂▁▁▃▅▃▄▅▇▆▇▆▇▇█▇▆▇█▇██▇▇▇███
train/balanced_acc,▂▃▁▂▂▃▃▃▃▃▄▄▄▄▅▆▆▆▆▅▅▇▇▆█▇▇▇▇▅▆▇▆▇█▇▇█▇▆
train/loss,████▇▇▇▇▇▇▆▆▅▅▄▃▄▂▃▃▃▃▁▂▂▂▁▃▄▂▂▃▂▂▁▃▂▁▂▂
train/macro_auc,▁▂▂▂▂▃▄▄▅▆▄▅▇▆▇▇▇▇▇▇███▇▆▇▇▇▇█▇▆█▇▇▆▇██▇
train/macro_f1,▂▃▄▄▂▃▅▃▄▃▃▂▂▁▃▃▄▄▅▄█▆▇▆▇▇▇▇▅▇▇▇▇▇▇▆▆▇▆▇
val/accuracy,▇▇▇▇▇▅▂▁▃▃▅▅▅▆▆▆▇▇▇▇█▇▇▇████████████████
val/balanced_acc,▁▅▄▃▃▃▃▄▄▅▆▇▆▇▇▇██▇▇████████████████████
+5,...


In [31]:
prediction_df

,case_id,tooth_number,label,prediction,prob_healed,prob_not_healed
0,DSA144pre,30,0,0,0.517243,0.482757
1,DSA135pre,4,0,0,0.516725,0.483275
2,DSA119pre,15,0,1,0.478940,0.521060
3,DSA067pre,15,1,1,0.480524,0.519476
4,DSA048pre,10,0,0,0.524965,0.475035
5,DSA-108PRE,31,0,1,0.446299,0.553701
6,DSA154pre,3,1,0,0.523801,0.476199
7,DSA211pre,15,0,1,0.439161,0.560839
8,DSA036pre,3,1,1,0.467484,0.532516
9,DSA199pre,5,0,0,0.579698,0.420302


In [33]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
)


# ============================================================
# 1. SELECT THRESHOLD USING VALIDATION SET ONLY
# ============================================================

val_y_true = val_result["y_true"]
val_prob_not_healed = val_result["y_prob"][:, 1]

thresholds = np.linspace(
    0.20,
    0.80,
    50,
)

best_threshold = None
best_bal_acc = -np.inf

for threshold in thresholds:

    val_pred = (
        val_prob_not_healed
        >= threshold
    ).astype(int)

    bal_acc = balanced_accuracy_score(
        val_y_true,
        val_pred,
    )

    if bal_acc > best_bal_acc:
        best_bal_acc = bal_acc
        best_threshold = threshold


print(
    f"Validation-selected threshold: "
    f"{best_threshold:.3f}"
)

print(
    f"Validation balanced accuracy: "
    f"{best_bal_acc:.4f}"
)


# ============================================================
# 2. APPLY THE FIXED VALIDATION THRESHOLD TO TEST SET
# ============================================================

test_y_true = test_result["y_true"]
test_prob_not_healed = test_result["y_prob"][:, 1]

test_pred_thresholded = (
    test_prob_not_healed
    >= best_threshold
).astype(int)


# ============================================================
# 3. TEST METRICS
# ============================================================

test_auc = roc_auc_score(
    test_y_true,
    test_prob_not_healed,
)

test_bal_acc = balanced_accuracy_score(
    test_y_true,
    test_pred_thresholded,
)

test_macro_f1 = f1_score(
    test_y_true,
    test_pred_thresholded,
    average="macro",
    zero_division=0,
)

test_recall = recall_score(
    test_y_true,
    test_pred_thresholded,
    labels=[0, 1],
    average=None,
    zero_division=0,
)

test_cm = confusion_matrix(
    test_y_true,
    test_pred_thresholded,
    labels=[0, 1],
)


print(
    f"\nTest AUC: {test_auc:.4f}"
)

print(
    f"Test balanced accuracy: "
    f"{test_bal_acc:.4f}"
)

print(
    f"Test macro F1: "
    f"{test_macro_f1:.4f}"
)

print("\nPer-class recall:")
print(
    f"Healed: "
    f"{test_recall[0]:.4f}"
)
print(
    f"Not-healed: "
    f"{test_recall[1]:.4f}"
)

print(
    "\nConfusion matrix:"
)
print(
    test_cm
)


# ============================================================
# 4. SAVE THRESHOLDED TEST PREDICTIONS
# ============================================================

thresholded_test_df = pd.DataFrame(
    {
        "case_id":
            test_result["case_ids"],

        "label":
            test_y_true,

        "prob_healed":
            test_result["y_prob"][:, 0],

        "prob_not_healed":
            test_prob_not_healed,

        "prediction_0.5":
            (
                test_prob_not_healed
                >= 0.5
            ).astype(int),

        "prediction_val_threshold":
            test_pred_thresholded,
    }
)

print(
    thresholded_test_df
)

Validation-selected threshold: 0.506
Validation balanced accuracy: 0.7421

Test AUC: 0.5789
Test balanced accuracy: 0.5842
Test macro F1: 0.4497

Per-class recall:
Healed: 0.3684
Not-healed: 0.8000

Confusion matrix:
[[ 7 12]
 [ 1  4]]
       case_id  label  prob_healed  prob_not_healed  prediction_0.5  \
0    DSA144pre      0     0.517243         0.482757               0   
1    DSA135pre      0     0.516725         0.483275               0   
2    DSA119pre      0     0.478940         0.521060               1   
3    DSA067pre      1     0.480524         0.519476               1   
4    DSA048pre      0     0.524965         0.475035               0   
5   DSA-108PRE      0     0.446299         0.553701               1   
6    DSA154pre      1     0.523801         0.476199               0   
7    DSA211pre      0     0.439161         0.560839               1   
8    DSA036pre      1     0.467484         0.532516               1   
9    DSA199pre      0     0.579698         0.420302   